In [60]:

from collections import Counter, deque, defaultdict
import networkx as nx
import matplotlib.pyplot as plt

S = "S"
O = "O"
DIVIDE = "/"
DASH = "-"
EQ = "="
M = "M"
W = "W"
Z = "Z"
T = "T"  # Define terminal node

MAX_DEPTH = 7
MAX_CONTIG_N = 3  # No more than 3 contiguous N's (N is W/Z/M)
N_VALS = (W, Z, M)

def is_N(symbol):
    return symbol in N_VALS

def count_trailing_N(path):
    cnt = 0
    for tok in reversed(path):
        if is_N(tok):
            cnt += 1
        else:
            break
    return cnt

def count_max_contiguous_N(path):
    max_n = 0
    curr = 0
    for tok in path:
        if is_N(tok):
            curr += 1
            max_n = max(max_n, curr)
        else:
            curr = 0
    return max_n

def contains_illegal_wwz(path):
    """
    Returns True if any W->W with a Z child branch is not followed immediately
    by [W, O, /, -, =]. This prevents illegal WWZ branching as per the rules.
    """
    for i in range(len(path)-2):
        if path[i] == W and path[i+1] == W and path[i+2] == Z:
            # If W->W->Z, this is not legal: W's after W must either be W, O, /, -, =
            return True
    return False

def has_illegal_z_dash_eq(path):
    """
    Returns True if there is a Z->DASH->EQ sequence.
    """
    for i in range(len(path)-2):
        if path[i] == Z and path[i+1] == DASH and path[i+2] == EQ:
            return True
    return False

def can_terminate_inverted(path):
    """
    See the new rules:
      - Cannot terminate immediately after 'S' ([S, T]) - S must be followed by a number (N=W/Z/M).
      - Cannot terminate on W->Z (must go W->Z->W, so W->Z->T is not allowed).
      - Cannot terminate on ...Z->-, so [-2]==Z and [-1]==DASH.
      - Absolutely NO >3 contiguous N's anywhere in the path.
      - W->W can only be followed by W or O,/,-,=.
      - Only allow termination if last symbol is not O, =, or /.
      - Cannot terminate (or generally allow) Z->DASH->EQ: after Z->DASH, must have N next, not EQ.
      - (Legacy rules preserved where relevant)
    """
    if not path or len(path) < 2:
        return False
    last = path[-1]

    # S cannot terminate directly ([S, T]) or [S, ... T] if not at least one N after S
    if path == [S]:
        return False
    if path[0] == S:
        if len(path) == 2 and path[1] == T:
            return False
        # S must be followed by a N
        if len(path) >= 2 and not is_N(path[1]) and path[1] != T:
            return False
        # Do not allow [S, T]
        if len(path) == 2 and path[1] == T:
            return False

    # No >3 contiguous N's anywhere
    if count_max_contiguous_N(path) > MAX_CONTIG_N:
        return False

    # Cannot terminate if last symbol is O, =, or /
    if last in (O, EQ, DIVIDE):
        return False

    # W->Z cannot terminate: must go W->Z->W...
    if len(path) >= 2 and path[-2] == W and path[-1] == Z:
        return False

    # --- NEW RULE: cannot terminate ... -> Z -> - (i.e., last two are [Z, DASH])
    if len(path) >= 2 and path[-2] == Z and path[-1] == DASH:
        return False

    # Block if any Z->DASH->EQ sequence in the path
    if has_illegal_z_dash_eq(path):
        return False

    # If W->W->Z pattern anywhere, forbid
    if contains_illegal_wwz(path):
        return False

    # W->W can only be followed by W or O, /, -, =
    # (Enforced in tree expansion, but double-check here)
    for i in range(len(path)-2):
        if path[i] == W and path[i+1] == W and path[i+2] not in (W, O, DIVIDE, DASH, EQ):
            return False

    # ---- (Keep other legacy rules) ----

    # 1. S->M=W and =->M=W (no S M W or = M W at end)
    if len(path) >= 3 and (
        (path[-3] == S and path[-2] == M and path[-1] == W) or
        (path[-3] == EQ and path[-2] == M and path[-1] == W)
    ):
        return False
    # 2. S->W=M and S->M=W
    if len(path) >= 3 and (
        (path[-3] == S and path[-2] == W and path[-1] == M) or
        (path[-3] == S and path[-2] == M and path[-1] == W)
    ):
        return False
    # 3. S->W_x = W_y x != y and = -> W_x = W_y
    if EQ in path:
        eq_idx = path.index(EQ)
        # Go left from = for W streak
        left_w = 0
        i = eq_idx - 1
        while i >= 0 and path[i] == W:
            left_w += 1
            i -= 1
        # Go right from = for W streak
        right_w = 0
        i = eq_idx + 1
        while i < len(path) and path[i] == W:
            right_w += 1
            i += 1
        if left_w > 0 and right_w > 0 and left_w != right_w:
            if S in path[:eq_idx-left_w+1]:
                return False
        if eq_idx == 0 and left_w > 0 and right_w > 0 and left_w != right_w:
            return False

    # 4. S->W(1 or 3 in a row)=M and =->W(1 or 3 in a row)=M
    def check_w_streak_before_eq_and_M(path):
        if len(path) >= 4:
            if (path[-4] == S and path[-3] == W and path[-2] == EQ and path[-1] == M) or \
               (path[-4] == EQ and path[-3] == W and path[-2] == EQ and path[-1] == M):
                w_streak = 0
                i = -4
                while abs(i) <= len(path) and path[i] == W:
                    w_streak += 1
                    i -= 1
                if w_streak in (1, 3):
                    return True
        return False

    if check_w_streak_before_eq_and_M(path):
        return False

    def check_M_before_eq_and_W_streak(path):
        if EQ in path:
            eq_idx = path.index(EQ)
            if eq_idx >= 2:
                if path[eq_idx - 2] == S and path[eq_idx - 1] == M:
                    w_streak = 0
                    for tok in path[eq_idx+1:]:
                        if tok == W:
                            w_streak += 1
                        else:
                            break
                    if w_streak in (1,3):
                        return True
                if path[0] == EQ and path[1] == M:
                    w_streak = 0
                    for tok in path[eq_idx+1:]:
                        if tok == W:
                            w_streak += 1
                        else:
                            break
                    if w_streak in (1,3):
                        return True
        return False

    if check_M_before_eq_and_W_streak(path):
        return False

    # 5. S -> N-=N and S -> N=N- (N is stream of M/W/0/Z), or same with = at root
    def is_stream(n):
        return n in (M,W,Z,0,'0')

    if len(path) >= 4:
        # S -> N - = N
        if path[-4] == S and is_stream(path[-3]) and path[-2] == DASH and path[-1] == EQ:
            return False
        # S -> N = N -
        if path[-4] == S and is_stream(path[-3]) and path[-2] == EQ and path[-1] == DASH:
            return False
        # = -> N - = N
        if path[-4] == EQ and is_stream(path[-3]) and path[-2] == DASH and path[-1] == EQ:
            return False
        # = -> N = N -
        if path[-4] == EQ and is_stream(path[-3]) and path[-2] == EQ and path[-1] == DASH:
            return False

    return True

def build_inverted_tree(level_filter_arr=None):
    """
    Build a TREE with these rules:
      - No more than MAX_CONTIG_N=3 contiguous Ns (N is W/Z/M) *ANYWHERE* in the path.
      - If the current path would create >3 contiguous Ns *anywhere*, do not branch or terminate here.
      - If you have hit 3 contiguous Ns (trailing), you must branch next to a non-N (O, /, -, =, T).
      - Z -> Z is allowed, BUT after Z->Z you must have W as the next child; and that W must be forced to immediately have as children only O,/, -, = (not N)
      - Z -> W is allowed, but after Z->W, the W **must** be immediately forced to have as children only O, /, -, = (not N)
      - W->W can only be followed by W or O,/,-,=
      - You cannot terminate after S ([S,T]), S must be followed by one N
      - You cannot terminate on W->Z, must go W->Z->W only
      - You cannot terminate on ...Z -> -
      - Z->DASH->EQ is forbidden: after Z->DASH, you can only have an N (not EQ, not O, etc)

    level_filter_arr: Optional list/array of length 16. Index is level. 
        If entry is not None/empty ("") at index k, then only allow nodes at depth k whose value is level_filter_arr[k].
        (Index 0 must be 'S'.)
    """
    # Defensive checks for the filter array
    if level_filter_arr is not None:
        if len(level_filter_arr) != 16:
            raise ValueError("level_filter_arr must be length 16")
        if not (level_filter_arr[0] == S):
            raise ValueError("level_filter_arr[0] must be 'S'")

    G = nx.DiGraph()
    node_labels = {}

    def get_node_id(path):
        return tuple(path)

    root_path = [S]
    root_nodeid = get_node_id(root_path)
    G.add_node(root_nodeid, label=S)
    node_labels[root_nodeid] = S

    queue = deque()
    queue.append((root_path,))

    while queue:
        (path,) = queue.popleft()
        cur_symbol = path[-1]
        cur_nodeid = get_node_id(path)
        depth = len(path) - 1

        # --- PRUNE using level_filter_arr restriction ---
        if level_filter_arr is not None:
            if depth < len(level_filter_arr):
                expected_val = level_filter_arr[depth]
                # Note: index 0 is always S (already checked at root construction)
                if expected_val is not None and expected_val != "" and cur_symbol != expected_val:
                    continue

        # HARD CUT: If this path has >3 contiguous N's anywhere, do not expand it further
        if count_max_contiguous_N(path) > MAX_CONTIG_N:
            continue

        # HARD CUT: block any path that contains illegal Z->D->EQ pattern anywhere
        if has_illegal_z_dash_eq(path):
            continue

        # Cannot terminate after [S], must be at least [S, N] before considering ending
        if depth >= MAX_DEPTH:
            if can_terminate_inverted(path):
                # Also block if last two are [Z, DASH] (terminate after Z- is forbidden)
                if not (len(path) >= 2 and path[-2] == Z and path[-1] == DASH):
                    t_path = path + [T]
                    t_id = get_node_id(t_path)
                    G.add_node(t_id, label=T)
                    node_labels[t_id] = T
                    G.add_edge(cur_nodeid, t_id)
            continue

        if can_terminate_inverted(path):
            # Also block if last two are [Z, DASH] (terminate after Z- is forbidden)
            if not (len(path) >= 2 and path[-2] == Z and path[-1] == DASH):
                t_path = path + [T]
                t_id = get_node_id(t_path)
                G.add_node(t_id, label=T)
                node_labels[t_id] = T
                G.add_edge(cur_nodeid, t_id)

        trailing_N = count_trailing_N(path)
        if trailing_N >= MAX_CONTIG_N:
            # After 3 contiguous N's at the tail, only allow non-N's (O, /, -, =)
            possible_children = [O, DIVIDE, DASH, EQ]
        else:
            if cur_symbol == S:
                # S must be followed by only [W, Z, M]
                possible_children = [W, Z, M]
            elif cur_symbol == O:
                possible_children = [W, Z, M]
            elif cur_symbol == DIVIDE:
                possible_children = [W, Z, M]
            elif cur_symbol == DASH:
                # If the parent is Z, then after Z->DASH, can only have N (W/Z/M), not EQ, nor O, nor '/'
                if len(path) >= 2 and path[-2] == Z:
                    possible_children = [W, Z, M]
                else:
                    possible_children = [W, Z, M, EQ]
            elif cur_symbol == EQ:
                possible_children = [W, Z, M]
            elif cur_symbol == M:
                possible_children = [O, DIVIDE, DASH, EQ]
            elif cur_symbol == W:
                # W->W can only be followed by W or O,/,-,=, so filter accordingly
                prev_symbol = path[-2] if len(path) >= 2 else None
                if prev_symbol == W:
                    possible_children = [W, O, DIVIDE, DASH, EQ]
                else:
                    possible_children = [Z, W, O, DIVIDE, DASH, EQ]
            elif cur_symbol == Z:
                # Z->Z, Z->W, Z->O, Z->-, Z->=
                possible_children = [Z, W, O, DASH, EQ]
            else:
                possible_children = []
        # For root S, always only [W, Z, M]
        if cur_symbol == S:
            possible_children = [W, Z, M]

        for child in possible_children:
            # --- PRUNE using level_filter_arr restriction for the child node ---
            # The level (depth+1) is for the next child node.
            if level_filter_arr is not None:
                child_level = depth + 1
                if child_level < len(level_filter_arr):
                    expected_val = level_filter_arr[child_level]
                    if expected_val is not None and expected_val != "" and child != expected_val:
                        continue

            # --- Z->DASH->EQ restriction: If expanding Z->DASH, do not allow EQ after
            # (already handled in possible_children above, so EQ is omitted if cur is DASH and prev is Z)
            # --- Z SPECIAL LOGIC ---
            # Z->W: W must be forced to only O, /, -, =
            if cur_symbol == Z and child == W:
                child_path = path + [child]
                if count_max_contiguous_N(child_path) > MAX_CONTIG_N:
                    continue
                # Also block if creating Z->DASH->EQ with this expansion
                # (but W can't be EQ, so safe)
                child_id = get_node_id(child_path)
                if child_id not in G:
                    G.add_node(child_id, label=child)
                    node_labels[child_id] = child
                    # Force next children for W: O, /, -, =
                    for forced_after in [O, DIVIDE, DASH, EQ]:
                        forced_child_level = depth + 2
                        if level_filter_arr is not None:
                            if forced_child_level < len(level_filter_arr):
                                forced_val = level_filter_arr[forced_child_level]
                                if forced_val is not None and forced_val != "" and forced_after != forced_val:
                                    continue
                        forced_path = child_path + [forced_after]
                        if count_max_contiguous_N(forced_path) > MAX_CONTIG_N:
                            continue
                        # Forbid W->W->Z pattern at the expansion phase
                        if len(child_path) >= 2 and child_path[-2] == W and forced_after == Z:
                            continue
                        # For Z->DASH->EQ restriction: do not allow Z->DASH->EQ at all
                        if len(forced_path) >= 3 and forced_path[-3] == Z and forced_path[-2] == DASH and forced_path[-1] == EQ:
                            continue
                        forced_id = get_node_id(forced_path)
                        if forced_id not in G:
                            G.add_node(forced_id, label=forced_after)
                            node_labels[forced_id] = forced_after
                            queue.append((forced_path,))
                        G.add_edge(child_id, forced_id)
                G.add_edge(cur_nodeid, child_id)
                continue

            # Z->Z: after this, must immediately add W, and then W only gets O,/, -, =
            if cur_symbol == Z and child == Z:
                child_path = path + [child]   # path + [Z]
                if count_max_contiguous_N(child_path) > MAX_CONTIG_N:
                    continue
                child_id = get_node_id(child_path)
                if child_id not in G:
                    G.add_node(child_id, label=child)
                    node_labels[child_id] = child
                    # Only one expansion: immediately W as next
                    zz_w_path = child_path + [W]
                    forced_child_level = depth + 2
                    if level_filter_arr is not None:
                        if forced_child_level < len(level_filter_arr):
                            forced_val = level_filter_arr[forced_child_level]
                            if forced_val is not None and forced_val != "" and W != forced_val:
                                # If not permitted at this level, skip this W branch
                                G.add_edge(child_id, child_id)  # Make sure the edge exists, even if no forced W (no expansion)
                                G.add_edge(cur_nodeid, child_id)
                                continue
                    if count_max_contiguous_N(zz_w_path) > MAX_CONTIG_N:
                        continue
                    zz_w_id = get_node_id(zz_w_path)
                    if zz_w_id not in G:
                        G.add_node(zz_w_id, label=W)
                        node_labels[zz_w_id] = W
                        # Force W's children: O,/, -, =
                        for forced_after in [O, DIVIDE, DASH, EQ]:
                            zz_forced_child_level = depth + 3
                            if level_filter_arr is not None:
                                if zz_forced_child_level < len(level_filter_arr):
                                    forced_w_val = level_filter_arr[zz_forced_child_level]
                                    if forced_w_val is not None and forced_w_val != "" and forced_after != forced_w_val:
                                        continue
                            forced_path = zz_w_path + [forced_after]
                            if count_max_contiguous_N(forced_path) > MAX_CONTIG_N:
                                continue
                            # Forbid W->W->Z at the expansion phase
                            if len(zz_w_path) >= 2 and zz_w_path[-2] == W and forced_after == Z:
                                continue
                            # For Z->DASH->EQ restriction: do not allow Z->DASH->EQ at all
                            if len(forced_path) >= 3 and forced_path[-3] == Z and forced_path[-2] == DASH and forced_path[-1] == EQ:
                                continue
                            forced_id = get_node_id(forced_path)
                            if forced_id not in G:
                                G.add_node(forced_id, label=forced_after)
                                node_labels[forced_id] = forced_after
                                queue.append((forced_path,))
                            G.add_edge(zz_w_id, forced_id)
                    G.add_edge(child_id, zz_w_id)
                G.add_edge(cur_nodeid, child_id)
                continue

            # Expansion: for W->W, do not allow W->W->Z (illegal)
            if cur_symbol == W and child == W:
                prev_symbol = path[-2] if len(path) >= 2 else None
                # Do not allow W->W->Z
                # Since the loop follows with child_path, prevent child_path+[Z] as a direct child
                child_path = path + [child]
                if count_max_contiguous_N(child_path) > MAX_CONTIG_N:
                    continue
                # For Z->DASH->EQ restriction, it's impossible here since W->W can't yield that
                child_id = get_node_id(child_path)
                if child_id not in G:
                    G.add_node(child_id, label=child)
                    node_labels[child_id] = child
                    queue.append((child_path,))
                G.add_edge(cur_nodeid, child_id)
                # When expanding W->W, W's children must only be W or O,/,-,=.
                # This is already enforced in possible_children for prev_symbol == W, so skip special logic here.
                continue

            # Forbid W->W->Z in general by filtering child_path
            if len(path) >= 1 and path[-1] == W and child == Z:
                prev_symbol = path[-2] if len(path) >= 2 else None
                if prev_symbol == W:
                    # W->W->Z is illegal
                    continue

            # W->Z: do not allow immediate termination, must add forced W as next step
            if cur_symbol == W and child == Z:
                # Only allow W->Z->W, not W->Z->T or W->Z->non-W
                zw_path = path + [child]  # path + [Z]
                # Must not create >3 contiguous N
                if count_max_contiguous_N(zw_path) > MAX_CONTIG_N:
                    continue
                zw_id = get_node_id(zw_path)
                if zw_id not in G:
                    G.add_node(zw_id, label=Z)
                    node_labels[zw_id] = Z
                    # Only one forced child, W
                    zw_w_path = zw_path + [W]
                    forced_child_level = depth + 2
                    if level_filter_arr is not None:
                        if forced_child_level < len(level_filter_arr):
                            forced_val = level_filter_arr[forced_child_level]
                            if forced_val is not None and forced_val != "" and W != forced_val:
                                G.add_edge(zw_id, zw_id)
                                G.add_edge(cur_nodeid, zw_id)
                                continue
                    if count_max_contiguous_N(zw_w_path) > MAX_CONTIG_N:
                        continue
                    zw_w_id = get_node_id(zw_w_path)
                    if zw_w_id not in G:
                        G.add_node(zw_w_id, label=W)
                        node_labels[zw_w_id] = W
                        queue.append((zw_w_path,))
                    G.add_edge(zw_id, zw_w_id)
                G.add_edge(cur_nodeid, zw_id)
                continue

            # All other moves, normal logic
            child_path = path + [child]
            # Forbid W->W->Z at expansion time
            if len(path) >= 1 and path[-1] == W and child == Z:
                prev_symbol = path[-2] if len(path) >= 2 else None
                if prev_symbol == W:
                    continue
            if count_max_contiguous_N(child_path) > MAX_CONTIG_N:
                continue

            # For Z->DASH->EQ restriction: If the last three tokens in the new child path are Z->DASH->EQ,
            # skip that expansion entirely.
            if len(child_path) >= 3 and child_path[-3] == Z and child_path[-2] == DASH and child_path[-1] == EQ:
                continue

            #--- New: prevent creating ... -> Z -> DASH -> T
            # We'll prevent adding a child == T to any node whose previous 2 are [Z, DASH]
            # But this is already handled in can_terminate_inverted and in the T-adding spot above!
            child_id = get_node_id(child_path)
            if child_id not in G:
                G.add_node(child_id, label=child)
                node_labels[child_id] = child
                queue.append((child_path,))
            G.add_edge(cur_nodeid, child_id)

    return G, node_labels

# Example usage (original usage for backward compatibility)
# Build and display the tree

# Example usage with a valid 16-length array sample for level_filter_arr:
sample_level_filter = [S, Z, None, W, None, W, None, None, None, None, None, None, None, None, None, None]  # 16 elements
inverted_G, inverted_labels = build_inverted_tree(sample_level_filter)


# Node counts and T's for debug
type_cnt_inv = Counter()
for nodeid in inverted_G.nodes:
    label = inverted_G.nodes[nodeid].get("label", "")
    type_cnt_inv[label] += 1
num_terminators_inv = type_cnt_inv.get(T, 0)

print("Node counts in inverted TREE:", type_cnt_inv)
print("Number of terminal nodes (T):", num_terminators_inv)
print("Total nodes in inverted TREE:", len(inverted_G.nodes))

# Uncomment for visualization if needed
# try:
#     inv_pos = nx.nx_pydot.graphviz_layout(inverted_G, prog='dot')
# except Exception:
#     inv_pos = nx.spring_layout(inverted_G)

# def show_path_label(node_tuple):
#     symbol = node_tuple[-1]
#     depth = len(node_tuple) - 1
#     return f"{symbol}\ndepth={depth}"

# draw_labels_inv = {k: show_path_label(k) for k in inverted_G.nodes}

# plt.figure(figsize=(20, 14))
# nx.draw(inverted_G, inv_pos, with_labels=True, labels=draw_labels_inv, arrows=False, node_size=200, node_color='lightyellow', font_size=10)
# plt.title("Inverted S → N TREE (N=W,Z,M), max 3 contiguous N, otherwise legacy expansion")
# plt.tight_layout(pad=3.0)
# plt.show()

# plt.show()



Node counts in inverted TREE: Counter({'T': 372, 'W': 166, 'Z': 109, 'M': 88, '=': 65, 'O': 43, '-': 43, '/': 42, 'S': 1})
Number of terminal nodes (T): 372
Total nodes in inverted TREE: 929


In [61]:
def get_terminal_paths_tree(tree, T, start_node=None):
    """
    Find all paths from the root to terminal nodes labeled T in a tree,
    returning paths as lists of labels.
    If start_node is not given, starts from the root (the unique node with in-degree 0).
    Assumes 'tree' is a directed tree (no cycles, unique parent for all nodes except root).
    """
    import networkx as nx

    if start_node is None:
        # In a tree, the root has in-degree 0
        roots = [n for n in tree.nodes if tree.in_degree(n) == 0]
        if not roots:
            raise ValueError("No root found in the tree.")
        start_node = roots[0]

    results = []

    def dfs(node, path):
        node_label = tree.nodes[node].get("label", "")
        new_path = path + [node_label]
        children = list(tree.successors(node))
        if not children:
            if node_label == T:
                results.append(new_path)
        else:
            for child in children:
                dfs(child, new_path)

    dfs(start_node, [])

    return results

terminal_paths_tree = get_terminal_paths_tree(inverted_G, T)
for p in terminal_paths_tree:
    print(p)
print(len(terminal_paths_tree))


['S', 'Z', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'Z', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'W', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'W', '-', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'O', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'O', 'Z', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', 'O', 'M', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '/', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '/', 'Z', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '/', 'M', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '-', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '-', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '-', 'Z', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '-', 'M', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '=', 'W', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '=', 'Z', 'T']
['S', 'Z', 'Z', 'W', 'O', 'W', '=', 'M', 'T']
['S', 'Z', 'Z', 'W', '/', 'W', 'T']
['S', 'Z', 'Z', 'W', '/', 'W', 'Z', 'W', 'T']
['S', 'Z', 'Z', 'W', '/', 'W', 'W', 'T']
['S', 'Z', 'Z', 'W', '/', 'W', 'W', 'W', 'T']
['S', '

In [46]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter, deque

S = "S"
O = "O"
DIVIDE = "/"
DASH = "-"
EQ = "="
M = "M"
W = "W"
Z = "Z"
T = "T"

MAX_DEPTH = 8
MAX_CONTIG_W = 3

# --- your can_terminate stays EXACTLY the same ---
def can_terminate(path):
    if len(path) == 1 and path[0] == S:
        return False

    if len(path) >= 4 and (path[-4:] == [S, M, EQ, W] or path[-4:] == [EQ, M, EQ, W]):
        return False

    if len(path) >= 4 and (path[-4:] == [S, W, EQ, M] or path[-4:] == [EQ, W, EQ, M]):
        return False

    if EQ in path:
        eq_idx = path.index(EQ)
        left_w = 0
        i = eq_idx - 1
        while i >= 0 and path[i] == W:
            left_w += 1
            i -= 1
        right_w = 0
        i = eq_idx + 1
        while i < len(path) and path[i] == W:
            right_w += 1
            i += 1
        if left_w > 0 and right_w > 0 and left_w != right_w:
            if S in path[:eq_idx - left_w + 1]:
                return False
        if len(path) >= eq_idx + 2:
            if path[0] == EQ and left_w > 0 and right_w > 0 and left_w != right_w:
                return False

    if EQ in path:
        eq_idx = path.index(EQ)
        left_w = 0
        i = eq_idx - 1
        while i >= 0 and path[i] == W:
            left_w += 1
            i -= 1
        if left_w in (1, 3) and eq_idx + 1 < len(path) and path[eq_idx + 1] == M:
            if S in path[:eq_idx - left_w + 1]:
                return False
            if path[0] == EQ:
                return False

    if len(path) >= 4 and ((path[-4] == S and path[-3] == M and path[-2] == EQ) or
                           (path[-4] == EQ and path[-3] == M and path[-2] == EQ)):
        i = len(path) - 1
        w_streak = 0
        while i > -1 and path[i] == W:
            w_streak += 1
            i -= 1
        if w_streak in (1, 3):
            return False

        eq_idx = len(path) - 2
        w_after_eq = 0
        for tok in path[eq_idx + 1:]:
            if tok == W:
                w_after_eq += 1
            else:
                break
        if w_after_eq in (1, 3):
            return False

    if len(path) >= 5:
        if path[-5] == S and path[-4] == DASH and path[-2] == EQ:
            n1_seq = []
            i = -3
            while abs(i) < len(path) and path[i] in (M, W, Z, '0', 0):
                n1_seq.append(path[i])
                i -= 1
            if len(n1_seq) >= 1:
                n2_seq = []
                i = -1
                while abs(i) <= len(path) and path[i] in (M, W, Z, '0', 0):
                    n2_seq.append(path[i])
                    i -= 1
                if len(n2_seq) >= 1:
                    return False

        if path[-5] == EQ and path[-4] == DASH and path[-2] == EQ:
            n1_seq = []
            i = -3
            while abs(i) < len(path) and path[i] in (M, W, Z, '0', 0):
                n1_seq.append(path[i])
                i -= 1
            if len(n1_seq) >= 1:
                n2_seq = []
                i = -1
                while abs(i) <= len(path) and path[i] in (M, W, Z, '0', 0):
                    n2_seq.append(path[i])
                    i -= 1
                if len(n2_seq) >= 1:
                    return False

        if path[-5] == S and path[-3] == EQ and path[-2] == DASH:
            n1_seq = []
            i = -4
            while abs(i) < len(path) and path[i] in (M, W, Z, '0', 0):
                n1_seq.append(path[i])
                i -= 1
            if len(n1_seq) >= 1:
                n2_seq = []
                i = -1
                while abs(i) <= len(path) and path[i] in (M, W, Z, '0', 0):
                    n2_seq.append(path[i])
                    i -= 1
                if len(n2_seq) >= 1:
                    return False

        if path[-5] == EQ and path[-3] == EQ and path[-2] == DASH:
            n1_seq = []
            i = -4
            while abs(i) < len(path) and path[i] in (M, W, Z, '0', 0):
                n1_seq.append(path[i])
                i -= 1
            if len(n1_seq) >= 1:
                n2_seq = []
                i = -1
                while abs(i) <= len(path) and path[i] in (M, W, Z, '0', 0):
                    n2_seq.append(path[i])
                    i -= 1
                if len(n2_seq) >= 1:
                    return False

    if path[-1] in (O, EQ, DASH, DIVIDE):
        return False

    return True

def build_custom_tree(level_filter_arr=None):
    """
    TREE version of your DAG:
      - Node id is the FULL PATH (tuple), so same symbol at same depth is NOT shared.
      - All expansion rules + W-streak constraint remain the same.
      - New: Accepts 'level_filter_arr' as a 16-element array of symbols/null.
        If level_filter_arr[level] is not None/"", prune all nodes at this level (index)
        where symbol != level_filter_arr[level]. Index 0 is always S and enforced S.
    """
    G = nx.DiGraph()
    node_labels = {}

    def node_id(path):
        return tuple(path)  # <-- key change: path identity

    root_path = [S]
    root_id = node_id(root_path)
    G.add_node(root_id, label=S)
    node_labels[root_id] = S

    queue = deque()
    # queue items: (path, w_streak)
    queue.append((root_path, 0))

    while queue:
        path, w_streak = queue.popleft()
        depth = len(path) - 1
        nodetype = path[-1]
        cur_id = node_id(path)

        # ------ Prune nodes at given level if mismatch with level_filter_arr ------
        if level_filter_arr is not None:
            # Only examine for level 0..15
            if depth < len(level_filter_arr):
                filter_val = level_filter_arr[depth]
                # skip None or "", but enforce filter
                if filter_val not in (None, "") and nodetype != filter_val:
                    continue

        # Depth cutoff
        if depth >= MAX_DEPTH:
            if can_terminate(path):
                t_path = path + [T]
                t_id = node_id(t_path)
                if t_id not in G:
                    G.add_node(t_id, label=T)
                    node_labels[t_id] = T
                G.add_edge(cur_id, t_id)
            continue

        # Terminate edge if allowed
        if can_terminate(path):
            t_path = path + [T]
            t_id = node_id(t_path)
            if t_id not in G:
                G.add_node(t_id, label=T)
                node_labels[t_id] = T
            G.add_edge(cur_id, t_id)

        def add_child(child_symbol):
            # update W streak
            if child_symbol == W:
                new_ws = w_streak + 1
            else:
                new_ws = 0

            # W streak constraint
            if child_symbol == W and new_ws > MAX_CONTIG_W:
                return

            child_path = path + [child_symbol]
            child_id = node_id(child_path)

            # --- Also prune children before enqueue if they don't match the filter (for efficiency) ---
            if level_filter_arr is not None:
                next_depth = len(child_path) - 1
                if next_depth < len(level_filter_arr):
                    filter_val = level_filter_arr[next_depth]
                    if filter_val not in (None, "") and child_symbol != filter_val:
                        return

            if child_id not in G:
                G.add_node(child_id, label=child_symbol)
                node_labels[child_id] = child_symbol
                queue.append((child_path, new_ws))

            G.add_edge(cur_id, child_id)

        # Expansion rules (same as your DAG)
        if nodetype == S:
            add_child(DASH)
            add_child(M)
            add_child(W)
            add_child(Z)
        elif nodetype == DASH:
            add_child(M)
            add_child(W)
        elif nodetype == EQ:
            add_child(M)
            add_child(W)
            add_child(Z)
            add_child(DASH)
        elif nodetype == M:
            add_child(O)
            add_child(EQ)
            add_child(DIVIDE)
            add_child(DASH)
        elif nodetype == Z:
            # Z->Z only if previous was W (W->Z->Z)
            if len(path) >= 2 and path[-2] == W:
                add_child(Z)
            add_child(O)
            add_child(EQ)
            add_child(DIVIDE)
            add_child(DASH)
        elif nodetype == W:
            add_child(W)
            add_child(Z)
            add_child(O)
            add_child(EQ)
            add_child(DIVIDE)
            add_child(DASH)
        elif nodetype == O:
            add_child(M)
            add_child(W)
            add_child(Z)
        elif nodetype == DIVIDE:
            add_child(M)
            add_child(W)

    return G, node_labels

# EXAMPLE USAGE:
# To match the example in your inverted code above, for instance:
level_filter_arr = [S, Z, Z, W, None, W, None, None, None, None, None, None, None, None, None, None]
# You may use None or "" for empty slots.

# Example: no filter provided (original behavior)
custom_G, custom_labels = build_custom_tree(level_filter_arr)

type_cnt = Counter()
for nodeid in custom_G.nodes:
    type_cnt[custom_G.nodes[nodeid].get("label", "")] += 1

print("Node counts in custom TREE:", type_cnt)
print("Number of T nodes:", type_cnt.get(T, 0))
print("Total nodes:", len(custom_G.nodes))
print("Total edges:", len(custom_G.edges))

# try:
#     pos = nx.nx_pydot.graphviz_layout(custom_G, prog="dot")
# except Exception:
#     pos = nx.spring_layout(custom_G)

# draw_labels = {k: f"{custom_labels[k]}\n{list(k)}" for k in custom_G.nodes}

# plt.figure(figsize=(20, 14))
# nx.draw(custom_G, pos, with_labels=True, labels=draw_labels, arrows=False,
#         node_size=200, node_color="lightcyan", font_size=8)
# plt.title("Custom TREE (no merging): node id = full path")
# plt.tight_layout(pad=3.0)
# plt.show()


Node counts in custom TREE: Counter({'S': 1, 'Z': 1, 'T': 1})
Number of T nodes: 1
Total nodes: 3
Total edges: 2


In [52]:
from collections import defaultdict

input_mapping = {'O': 1, 'M': 1, 'W': 2, 'Z': 1, 'DASH': 1, 'EQ': 1, 'DIVIDE': 1}
types = sorted(input_mapping.keys())
N = sum(input_mapping.values())

def is_valid_side(cnt):
    return (cnt.get("M", 0) > 0) or (cnt.get("W", 0) > 0) or (cnt.get("Z", 0) > 0)

def total_count(cnt):
    return sum(cnt.values())

def canon_pair(left, right):
    """Canonicalize unordered sides: return (A,B) where A <= B lexicographically."""
    lt = tuple(left.get(t, 0) for t in types)
    rt = tuple(right.get(t, 0) for t in types)
    return (lt, rt) if lt <= rt else (rt, lt)

def tuple_to_dict(tup):
    return {types[i]: tup[i] for i in range(len(types)) if tup[i]}

results = set()

left = {t: 0 for t in types}
right = {t: 0 for t in types}

def backtrack(i, left_sum, right_sum):
    if i == len(types):
        # non-empty sides
        if left_sum == 0 or right_sum == 0:
            return

        # "less than size eight" => total used < N
        if left_sum + right_sum >= N:
            return

        # rule: each side must contain at least one of M/W/Z
        if not (is_valid_side(left) and is_valid_side(right)):
            return

        results.add(canon_pair(left, right))
        return

    t = types[i]
    avail = input_mapping[t]

    # choose how many of type t go to left and right (disjoint), leftovers allowed
    for l in range(avail + 1):
        for r in range(avail - l + 1):
            left[t] = l
            right[t] = r
            backtrack(i + 1, left_sum + l, right_sum + r)

    left[t] = 0
    right[t] = 0

backtrack(0, 0, 0)

# --- print grouped by (left_size, right_size) with sizes also unordered ---
groups = defaultdict(list)
for lt, rt in results:
    ls = sum(lt)
    rs = sum(rt)
    groups[(ls, rs)].append((lt, rt))

big_sum = 0
for (ls, rs) in sorted(groups.keys()):
    print(f"=== Split sizes {ls} + {rs} (unordered sides) : {len(groups[(ls, rs)])} ways ===")
    for lt, rt in sorted(groups[(ls, rs)]):
        ldict = tuple_to_dict(lt)
        rdict = tuple_to_dict(rt)
        lstr = ", ".join(f"{k}:{v}" for k, v in ldict.items())
        rstr = ", ".join(f"{k}:{v}" for k, v in rdict.items())
        print(f"Left({ls}): {{{lstr}}} | Right({rs}): {{{rstr}}}")
    print()
    big_sum += len(groups[(ls, rs)])

print("Sum of all unordered (LHS,RHS) pairs with total used < N:", big_sum)


=== Split sizes 1 + 1 (unordered sides) : 4 ways ===
Left(1): {Z:1} | Right(1): {W:1}
Left(1): {Z:1} | Right(1): {M:1}
Left(1): {W:1} | Right(1): {W:1}
Left(1): {W:1} | Right(1): {M:1}

=== Split sizes 1 + 2 (unordered sides) : 31 ways ===
Left(1): {Z:1} | Right(2): {W:2}
Left(1): {Z:1} | Right(2): {O:1, W:1}
Left(1): {Z:1} | Right(2): {M:1, W:1}
Left(1): {Z:1} | Right(2): {M:1, O:1}
Left(1): {Z:1} | Right(2): {EQ:1, W:1}
Left(1): {Z:1} | Right(2): {EQ:1, M:1}
Left(1): {Z:1} | Right(2): {DIVIDE:1, W:1}
Left(1): {Z:1} | Right(2): {DIVIDE:1, M:1}
Left(1): {Z:1} | Right(2): {DASH:1, W:1}
Left(1): {Z:1} | Right(2): {DASH:1, M:1}
Left(1): {W:1} | Right(2): {W:1, Z:1}
Left(1): {W:1} | Right(2): {O:1, Z:1}
Left(1): {W:1} | Right(2): {O:1, W:1}
Left(1): {W:1} | Right(2): {M:1, Z:1}
Left(1): {W:1} | Right(2): {M:1, W:1}
Left(1): {W:1} | Right(2): {M:1, O:1}
Left(1): {W:1} | Right(2): {EQ:1, Z:1}
Left(1): {W:1} | Right(2): {EQ:1, W:1}
Left(1): {W:1} | Right(2): {EQ:1, M:1}
Left(1): {W:1} | Right